# Electrode Geometry Measurements

Measures the actual shape of every electrode patch `trielec` cuts into a surface mesh, instead
of eyeballing screenshots. For each of the 64 electrodes this computes:

| Metric | Meaning |
|---|---|
| `n_faces` | Number of triangles making up the patch |
| `area` | Total triangle area of the patch (mm²) |
| `perimeter` | Length of the patch's outer boundary (mm) |
| `compactness` | `perimeter / sqrt(area)` -- **roundness score**. A perfect circle scores `2*sqrt(pi) ≈ 3.545`; anything higher means the patch deviates from round (spikes, notches, coarse faceting). |
| `min_quality` | Worst individual triangle's shape quality in the patch, `1.0` = equilateral, `-> 0` = degenerate sliver |

Run the cells top to bottom. Change `MESH_NAME` in the config cell to switch which test head to measure.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, r"C:\S.R\SCIBS\src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scibs.utilities.file import read_tri
from scibs.scripts.trielec import load_electrode_file, trielec, _triangle_quality

pd.set_option("display.precision", 4)


## 2. Configuration

`MESH_NAME` picks which `data/<name>.tri` + `data/<name>_elec_descr.txt` pair to measure.
`CLEAN_SLIVERS` toggles `trielec`'s sliver-cleanup pass -- run it both ways to see the effect
on the numbers below.

In [ ]:
MESH_NAME = "testhead"       # or "egghead"
CLEAN_SLIVERS = False        # set True to compare against the cleaned-up mesh

DATA_DIR = r"C:\S.R\SCIBS\data"
TRI_PATH = f"{DATA_DIR}\\{MESH_NAME}.tri"
ELEC_PATH = f"{DATA_DIR}\\{MESH_NAME}_elec_descr.txt"


## 3. Run `trielec` to imprint + label the electrodes

In [ ]:
verts, faces = read_tri(TRI_PATH)
electrode_pts, electrode_radii = load_electrode_file(ELEC_PATH)

final_verts, final_faces, face_labels, warp_lines = trielec(
    verts, faces, electrode_pts, electrode_radii, clean_slivers=CLEAN_SLIVERS,
)

n_electrodes = len(electrode_pts)
print(f"{MESH_NAME}: {len(final_verts)} verts, {len(final_faces)} faces, "
      f"{(face_labels != -1).sum()} labeled faces across {n_electrodes} electrodes")


## 4. Per-electrode geometry metrics

`compactness` and `min_quality` are the two numbers that actually caught the bugs we found
earlier: a spike or notch inflates `compactness` well above the ~3.545 circle minimum, and a
sliver triangle collapses `min_quality` toward 0 regardless of how the patch looks in aggregate.

In [ ]:
def _boundary_perimeter(verts, faces):
    """Sum of edge lengths that belong to exactly one triangle (the patch outline)."""
    edge_count = {}
    for f in faces:
        for a, b in ((f[0], f[1]), (f[1], f[2]), (f[2], f[0])):
            key = (int(a), int(b)) if a < b else (int(b), int(a))
            edge_count[key] = edge_count.get(key, 0) + 1
    boundary_edges = [k for k, v in edge_count.items() if v == 1]
    if not boundary_edges:
        return 0.0
    a_idx = np.array([e[0] for e in boundary_edges])
    b_idx = np.array([e[1] for e in boundary_edges])
    return float(np.linalg.norm(verts[a_idx] - verts[b_idx], axis=1).sum())


def _triangle_areas(verts, faces):
    v0, v1, v2 = verts[faces[:, 0]], verts[faces[:, 1]], verts[faces[:, 2]]
    return 0.5 * np.linalg.norm(np.cross(v1 - v0, v2 - v0), axis=1)


def electrode_geometry_table(verts, faces, labels, n_electrodes):
    quality = _triangle_quality(verts, faces)
    areas = _triangle_areas(verts, faces)

    rows = []
    for label in range(1, n_electrodes + 1):
        idx = np.where(labels == label)[0]
        if len(idx) == 0:
            rows.append(dict(electrode=label, n_faces=0, area=0.0, perimeter=0.0,
                              compactness=np.nan, min_quality=np.nan))
            continue
        patch_faces = faces[idx]
        area = float(areas[idx].sum())
        perimeter = _boundary_perimeter(verts, patch_faces)
        compactness = perimeter / np.sqrt(area) if area > 0 else np.inf
        rows.append(dict(
            electrode=label, n_faces=len(idx), area=area, perimeter=perimeter,
            compactness=compactness, min_quality=float(quality[idx].min()),
        ))
    return pd.DataFrame(rows).set_index("electrode")


CIRCLE_COMPACTNESS = 2 * np.sqrt(np.pi)  # ~3.5449, the theoretical minimum (perfect circle)

table = electrode_geometry_table(final_verts, final_faces, face_labels, n_electrodes)
table["compactness_vs_circle_pct"] = (table["compactness"] / CIRCLE_COMPACTNESS - 1) * 100
table.sort_values("compactness", ascending=False).head(15)


## 5. Summary stats

In [ ]:
print(f"Theoretical roundest-possible compactness (perfect circle): {CIRCLE_COMPACTNESS:.4f}\n")
print(table[["n_faces", "area", "compactness", "min_quality"]].describe())

SLIVER_THRESHOLD = 0.05
bad_slivers = table[table["min_quality"] < SLIVER_THRESHOLD]
print(f"\nElectrodes with a sliver triangle below quality {SLIVER_THRESHOLD}: "
      f"{len(bad_slivers)} / {n_electrodes}")
print(bad_slivers.index.tolist())


## 6. Visualize -- roundness and sliver quality per electrode

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

ordered = table.sort_values("compactness", ascending=False)
axes[0].bar(ordered.index.astype(str), ordered["compactness"], color="steelblue")
axes[0].axhline(CIRCLE_COMPACTNESS, color="red", linestyle="--", label="perfect circle")
axes[0].set_ylabel("compactness (lower = rounder)")
axes[0].set_title(f"{MESH_NAME}: roundness per electrode (clean_slivers={CLEAN_SLIVERS})")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=90, labelsize=6)

ordered_q = table.sort_values("min_quality")
axes[1].bar(ordered_q.index.astype(str), ordered_q["min_quality"], color="darkorange")
axes[1].axhline(SLIVER_THRESHOLD, color="red", linestyle="--", label=f"sliver threshold ({SLIVER_THRESHOLD})")
axes[1].set_ylabel("min triangle quality (0 = sliver, 1 = equilateral)")
axes[1].set_title("Worst triangle quality per electrode")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=90, labelsize=6)

plt.tight_layout()
plt.show()


## 7. Locate the worst electrodes on the head

Highlights the `N_WORST` least-round electrodes in red so you can see where on the head
they actually are, rather than just reading numbers.

In [ ]:
N_WORST = 5
worst_ids = table.sort_values("compactness", ascending=False).head(N_WORST).index.tolist()
print("Worst electrodes by compactness:", worst_ids)

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection="3d")

base_mask = face_labels == -1
ax.plot_trisurf(
    final_verts[:, 0], final_verts[:, 1], final_verts[:, 2],
    triangles=final_faces[base_mask], color="lightgray", alpha=0.3, shade=False,
)

worst_mask = np.isin(face_labels, worst_ids)
other_mask = (~base_mask) & (~worst_mask)

if other_mask.any():
    ax.plot_trisurf(
        final_verts[:, 0], final_verts[:, 1], final_verts[:, 2],
        triangles=final_faces[other_mask], color="steelblue", shade=False,
    )
if worst_mask.any():
    ax.plot_trisurf(
        final_verts[:, 0], final_verts[:, 1], final_verts[:, 2],
        triangles=final_faces[worst_mask], color="red", shade=False,
    )

ax.set_title(f"Red = {N_WORST} least-round electrodes ({MESH_NAME}, clean_slivers={CLEAN_SLIVERS})")
ax.set_box_aspect([1, 1, 1])
plt.show()


---
## Notes

- `compactness` catches shape defects (spikes/notches) that survive even when every triangle
  individually looks fine -- it's a whole-patch measurement.
- `min_quality` catches a single bad sliver triangle even when the patch's overall outline
  looks round -- it's a per-triangle measurement. The two metrics catch different failure
  modes, which is why both are here.
- If `bad_slivers` in section 5 is non-empty, re-run with `CLEAN_SLIVERS = True` in section 2
  and compare -- `trielec`'s `cleanup_slivers()` pass should drive `min_quality` for every
  electrode above the sliver threshold (see `trielec.py`'s `--clean-slivers` flag).